# MLP Hyperparameter Search

自動搜尋最佳超參數組合（5種模型架構 × learning_rate × weight_decay × dropout_rate）

每個組合跑 **300 epochs**，最後輸出 Test RMSE 最小的組合。

In [1]:
!git clone https://github.com/Anson-ntuim/DL-Final.git


Cloning into 'DL-Final'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 2.09 MiB | 9.85 MiB/s, done.


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
import itertools
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


## 全域設定與超參數網格

In [3]:
# ── 固定設定 ─────────────────────────────────
BATCH_SIZE  = 64
NUM_EPOCHS  = 300
TRAIN_RATIO = 0.8
SEED = 42

# ── 超參數搜尋空間 ────────────────────────────
LEARNING_RATES = [0.001, 0.0005, 0.0001]
WEIGHT_DECAYS  = [1e-3, 1e-4, 1e-5]
DROPOUT_RATES  = [0.1, 0.2, 0.3]

# ── 5 種模型架構 (hidden_dims 列表) ──────────
MODEL_CONFIGS = {
    'Model_A': [512, 256, 128, 64],
    'Model_B': [256, 128, 64],
    'Model_C': [1024, 512, 256, 128, 64],
    'Model_D': [512, 512, 256, 128],
    'Model_E': [256, 256, 128, 128, 64],
}

total_combos = len(MODEL_CONFIGS) * len(LEARNING_RATES) * len(WEIGHT_DECAYS) * len(DROPOUT_RATES)
print(f'總共搜尋組合數: {total_combos} 個（每個跑 {NUM_EPOCHS} epochs）')


總共搜尋組合數: 135 個（每個跑 300 epochs）


## 工具函式 & 資料前處理

In [4]:
def parse_iso8601_duration(duration_str):
    if not isinstance(duration_str, str):
        return 0
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, duration_str)
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s

def parse_published_at(dt_str):
    try:
        dt = datetime.fromisoformat(dt_str.replace('Z', '+00:00'))
        return dt.hour, dt.weekday()
    except Exception:
        return 0, 0

class StandardScaler:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8
        return self
    def transform(self, X):
        return (X - self.mean_) / self.std_

class YouTubeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [5]:
# ── 讀取資料 ─────────────────────────────────
data_dir = 'DL-Final/data'
files = os.listdir(data_dir)
csv_files  = [f for f in files if f.endswith('.csv')]
json_files = [f for f in files if f.endswith('.json')]

if csv_files:
    df = pd.read_csv(os.path.join(data_dir, csv_files[0]))
elif json_files:
    df = pd.read_json(os.path.join(data_dir, json_files[0]))
else:
    raise FileNotFoundError('找不到資料檔')

print(f'Loaded data shape={df.shape}')

# ── 特徵工程 ─────────────────────────────────
if 'category_id' in df.columns:
    df = df[df['category_id'] != 10].copy()

df['category_id'] = df['category_id'].astype(str)
df_encoded = pd.get_dummies(df, columns=['category_id'], prefix='cat')

if 'duration_iso8601' in df.columns:
    df_encoded['video_duration_sec'] = df['duration_iso8601'].apply(parse_iso8601_duration)

if 'published_at' in df.columns:
    df_encoded[['pub_hour', 'pub_weekday']] = df['published_at'].apply(
        lambda x: pd.Series(parse_published_at(str(x)))
    )
    df_encoded['pub_hour_sin']    = np.sin(2 * np.pi * df_encoded['pub_hour'] / 24.0)
    df_encoded['pub_hour_cos']    = np.cos(2 * np.pi * df_encoded['pub_hour'] / 24.0)
    df_encoded['pub_weekday_sin'] = np.sin(2 * np.pi * df_encoded['pub_weekday'] / 7.0)
    df_encoded['pub_weekday_cos'] = np.cos(2 * np.pi * df_encoded['pub_weekday'] / 7.0)

df_encoded['subscriber_count']    = np.log1p(df_encoded['subscriber_count'])
df_encoded['video_duration_sec']  = np.log1p(df_encoded['video_duration_sec'])
df_encoded['description_length']  = np.log1p(df_encoded['description_length'])
df_encoded['channel_view_count']  = np.log1p(df_encoded['channel_view_count'])
df_encoded['channel_video_count'] = np.log1p(df_encoded['channel_video_count'])

df['days_since_published'] = df['published_at'].apply(
    lambda x: (datetime.now(timezone.utc) -
               datetime.fromisoformat(str(x).replace('Z', '+00:00'))).days
    if 'T' in str(x) else 0
)
df_encoded['days_since_published'] = np.log1p(df['days_since_published'])

CAT_COLS     = [col for col in df_encoded.columns if col.startswith('cat_')]
NUM_COLS     = ['subscriber_count', 'video_duration_sec', 'tags_count', 'description_length',
                'pub_hour_sin', 'pub_hour_cos', 'pub_weekday_sin', 'pub_weekday_cos',
                'days_since_published', 'channel_view_count', 'channel_video_count']
FEATURE_COLS = CAT_COLS + NUM_COLS

X = df_encoded[FEATURE_COLS].values.astype(np.float32)
INPUT_DIM = X.shape[1]

view_col = 'view_count' if 'view_count' in df.columns else 'viewCount'
y = np.log1p(df[view_col].values.astype(np.float32))

print(f'Feature matrix shape: {X.shape}')
print(f'Target range: [{y.min():.2f}, {y.max():.2f}]')


Loaded data shape=(4867, 27)
Feature matrix shape: (4867, 25)
Target range: [0.00, 17.06]


In [6]:
# ── 固定 Train/Test split（所有組合共用同一份切割）─────────
np.random.seed(SEED)
torch.manual_seed(SEED)

n_total = len(X)
n_train = int(n_total * TRAIN_RATIO)
indices = np.random.permutation(n_total)
train_idx, test_idx = indices[:n_train], indices[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler  = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

train_dataset = YouTubeDataset(X_train, y_train)
test_dataset  = YouTubeDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')


Train: 3893 | Test: 974


## 模型定義

In [7]:
class myActivation(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x)


class myLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, pred, target):
        return torch.mean((pred - target) ** 2)


class FlexMLP(nn.Module):
    """
    可彈性設定層數與寬度的 MLP。
    hidden_dims: list of int，例如 [512, 256, 128, 64]
    """
    def __init__(self, input_dim, hidden_dims, dropout_rate=0.1):
        super().__init__()
        layers = []
        in_dim = input_dim
        for i, h_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(myActivation())
            if i < len(hidden_dims) - 1:
                layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)


print('模型類別定義完成')


模型類別定義完成


## 訓練函式

In [8]:
def train_and_evaluate(model_name, hidden_dims, lr, wd, dr, num_epochs=NUM_EPOCHS):
    torch.manual_seed(SEED)
    model = FlexMLP(INPUT_DIM, hidden_dims, dropout_rate=dr).to(device)
    criterion = myLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15
    )

    for epoch in range(num_epochs):
        model.train()
        losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(x_batch)
            loss   = criterion(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            losses.append(loss.item())
        if epoch % 10 == 0:
          avg_loss = np.mean(losses)
          scheduler.step(avg_loss)
          print(f'Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg_loss:.4f}')

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            all_preds.append(model(x_batch.to(device)).cpu())
            all_labels.append(y_batch.cpu())

    preds_t  = torch.cat(all_preds).squeeze()
    labels_t = torch.cat(all_labels).squeeze()
    rmse = torch.sqrt(torch.mean((preds_t - labels_t) ** 2)).item()
    return rmse


print('訓練函式定義完成')


訓練函式定義完成


## 超參數搜尋（Grid Search）

> ⏳ 共 135 組合 × 300 epochs，請耐心等待。有 GPU 會快很多。

In [ ]:
results = []

all_combos = list(itertools.product(
    MODEL_CONFIGS.items(),
    LEARNING_RATES,
    WEIGHT_DECAYS,
    DROPOUT_RATES
))

total = len(all_combos)
best_rmse   = float('inf')
best_config = None

SEP  = '─' * 62
SEP2 = '═' * 62

print(SEP2)
print(f'  🔍 超參數 Grid Search 開始  （共 {total} 個組合）')
print(SEP2)
print()

for i, ((model_name, hidden_dims), lr, wd, dr) in enumerate(all_combos):

    print(SEP)
    print(f'  [{i+1:3d}/{total}]  正在訓練...')
    print(f'  模型架構  : {model_name}  {hidden_dims}')
    print(f'  lr        : {lr}    wd : {wd:.0e}    dropout : {dr}')
    print(SEP)

    rmse = train_and_evaluate(model_name, hidden_dims, lr, wd, dr)

    is_new_best = rmse < best_rmse
    if is_new_best:
        best_rmse   = rmse
        best_config = {
            'model_name':    model_name,
            'hidden_dims':   str(hidden_dims),
            'learning_rate': lr,
            'weight_decay':  wd,
            'dropout_rate':  dr,
            'test_rmse':     rmse
        }

    results.append({
        'model_name':    model_name,
        'hidden_dims':   str(hidden_dims),
        'learning_rate': lr,
        'weight_decay':  wd,
        'dropout_rate':  dr,
        'test_rmse':     rmse
    })

    star = '  ★ NEW BEST!' if is_new_best else ''
    print(f'  結果  →  Test RMSE : {rmse:.4f}{star}')
    print(f'  目前最佳 RMSE : {best_rmse:.4f}'
          f'  ({best_config["model_name"]}'
          f'  lr={best_config["learning_rate"]}'
          f'  wd={best_config["weight_decay"]:.0e}'
          f'  dr={best_config["dropout_rate"]})')
    print()

print(SEP2)
print('  ✅  搜尋完畢！')
print(SEP2)
print()
print('  🏆  最佳組合')
print(f'  模型架構     : {best_config["model_name"]}  {best_config["hidden_dims"]}')
print(f'  learning_rate: {best_config["learning_rate"]}')
print(f'  weight_decay : {best_config["weight_decay"]}')
print(f'  dropout_rate : {best_config["dropout_rate"]}')
print(f'  Test RMSE    : {best_config["test_rmse"]:.4f}')
print(SEP2)

══════════════════════════════════════════════════════════════
  🔍 超參數 Grid Search 開始  （共 135 個組合）
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  [  1/135]  正在訓練...
  模型架構  : Model_A  [512, 256, 128, 64]
  lr        : 0.001    wd : 1e-03    dropout : 0.1
──────────────────────────────────────────────────────────────
Epoch   1/300 | Loss: 37.3855
Epoch  11/300 | Loss: 2.1219
Epoch  21/300 | Loss: 1.8918
Epoch  31/300 | Loss: 1.8306
Epoch  41/300 | Loss: 1.6524
Epoch  51/300 | Loss: 1.5913
Epoch  61/300 | Loss: 1.6000
Epoch  71/300 | Loss: 1.4861
Epoch  81/300 | Loss: 1.4937
Epoch  91/300 | Loss: 1.4799
Epoch 101/300 | Loss: 1.4136
Epoch 111/300 | Loss: 1.3793
Epoch 121/300 | Loss: 1.4364
Epoch 131/300 | Loss: 1.3216
Epoch 141/300 | Loss: 1.4486
Epoch 151/300 | Loss: 1.3496
Epoch 161/300 | Loss: 1.3833
Epoch 171/300 | Loss: 1.3135
Epoch 181/300 | Loss: 1.3190
Epoch 191/300 | Loss: 1.3546
Epoch 201/300 | Los

## 結果分析

In [ ]:
df_results = pd.DataFrame(results).sort_values('test_rmse').reset_index(drop=True)

print('=' * 70)
print('Top 10 最佳組合（按 Test RMSE 排序）')
print('=' * 70)
print(df_results.head(10).to_string(index=False))

print('\n' + '=' * 70)
print('🏆  最佳組合（Test RMSE 最小）')
print('=' * 70)
best = df_results.iloc[0]
print(f'  模型架構     : {best["model_name"]}  {best["hidden_dims"]}')
print(f'  learning_rate: {best["learning_rate"]}')
print(f'  weight_decay : {best["weight_decay"]}')
print(f'  dropout_rate : {best["dropout_rate"]}')
print(f'  Test RMSE    : {best["test_rmse"]:.4f}')
print('=' * 70)

df_results.to_csv('hyperparam_search_results.csv', index=False)
print('\n結果已儲存至 hyperparam_search_results.csv')


## 視覺化結果

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_avg = df_results.groupby('model_name')['test_rmse'].min().sort_values()
axes[0].barh(model_avg.index, model_avg.values, color='steelblue')
axes[0].set_xlabel('Best Test RMSE')
axes[0].set_title('Best RMSE by Model Architecture')

lr_avg = df_results.groupby('learning_rate')['test_rmse'].min()
axes[1].bar([str(lr) for lr in lr_avg.index], lr_avg.values, color='coral')
axes[1].set_xlabel('Learning Rate')
axes[1].set_title('Best RMSE by Learning Rate')

dr_avg = df_results.groupby('dropout_rate')['test_rmse'].min()
axes[2].bar([str(dr) for dr in dr_avg.index], dr_avg.values, color='mediumseagreen')
axes[2].set_xlabel('Dropout Rate')
axes[2].set_title('Best RMSE by Dropout Rate')

plt.suptitle('Hyperparameter Search Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('hyperparam_search_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('圖表已儲存至 hyperparam_search_plot.png')


## 用最佳組合重新訓練最終模型

In [ ]:
best_row        = df_results.iloc[0]
BEST_MODEL_NAME = best_row['model_name']
BEST_HIDDEN     = MODEL_CONFIGS[BEST_MODEL_NAME]
BEST_LR         = best_row['learning_rate']
BEST_WD         = best_row['weight_decay']
BEST_DR         = best_row['dropout_rate']

print(f'以最佳組合重新訓練 {NUM_EPOCHS} epochs...')
print(f'  {BEST_MODEL_NAME} {BEST_HIDDEN} | lr={BEST_LR} wd={BEST_WD} dr={BEST_DR}')

torch.manual_seed(SEED)
final_model = FlexMLP(INPUT_DIM, BEST_HIDDEN, dropout_rate=BEST_DR).to(device)
criterion   = myLoss()
optimizer   = optim.Adam(final_model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=15
)

for epoch in range(NUM_EPOCHS):
    final_model.train()
    losses = []
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out  = final_model(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step(np.mean(losses))
    if (epoch + 1) % 50 == 0:
        print(f'  Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train Loss: {np.mean(losses):.4f}')

final_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        all_preds.append(final_model(x_batch.to(device)).cpu())
        all_labels.append(y_batch)

preds_t  = torch.cat(all_preds).squeeze()
labels_t = torch.cat(all_labels).squeeze()
final_rmse = torch.sqrt(torch.mean((preds_t - labels_t) ** 2)).item()
final_mae  = torch.mean(torch.abs(preds_t - labels_t)).item()
final_mse  = torch.mean((preds_t - labels_t) ** 2).item()

print('\n' + '=' * 50)
print('最終模型評估結果（最佳超參數）')
print('=' * 50)
print(f'Test MSE  : {final_mse:.4f}')
print(f'Test MAE  : {final_mae:.4f}')
print(f'Test RMSE : {final_rmse:.4f}')
print('=' * 50)

torch.save(final_model.state_dict(), 'best_model_state_dict.pt')
print('模型已儲存至 best_model_state_dict.pt')


In [ ]:
import matplotlib.pyplot as plt

preds_np  = preds_t.numpy()
labels_np = labels_t.numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(labels_np, preds_np, alpha=0.3, s=8, color='steelblue')
mn, mx = labels_np.min(), labels_np.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect')
axes[0].set_xlabel('Actual log(1+view_count)')
axes[0].set_ylabel('Predicted log(1+view_count)')
axes[0].set_title('Predicted vs Actual (Best Model)')
axes[0].legend()

residuals = preds_np - labels_np
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white')
axes[1].axvline(0, color='k', linestyle='--')
axes[1].set_xlabel('Residual (Pred - Actual)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution (Best Model)')

plt.suptitle(
    f'Best: {BEST_MODEL_NAME} | lr={BEST_LR} wd={BEST_WD} dr={BEST_DR} | RMSE={final_rmse:.4f}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('best_model_prediction.png', dpi=120, bbox_inches='tight')
plt.show()
print('圖表已儲存至 best_model_prediction.png')
